## 1. Cargar los raster y revisar sus proyecciones

In [1]:

from pathlib import Path
from qgis.PyQt.QtWidgets import QFileDialog

# ==========================================================
# CELDA 1 — SELECCIONAR DIRECTORIO DEL PROYECTO
# ==========================================================

# Abrir ventana para seleccionar la carpeta raíz
directorio = QFileDialog.getExistingDirectory(
    None,
    "Seleccione la carpeta raíz del taller",
    str(Path.home())
)

if not directorio:
    raise RuntimeError(
        "No se seleccionó un directorio."
    )

PROJECT_DIR = Path(directorio).resolve()

# Definir rutas
RASTER_DIR = PROJECT_DIR / "data" / "raster"
SHAPES_DIR = PROJECT_DIR / "data" / "shapes"
OUTPUT_DIR = RASTER_DIR / "salidas_comparacion"

# Validar estructura del proyecto
for directory in [RASTER_DIR, SHAPES_DIR]:
    if not directory.is_dir():
        raise FileNotFoundError(
            f"No se encontró la carpeta requerida:\n{directory}"
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio configurado correctamente")
print(f"Proyecto: {PROJECT_DIR}")
print(f"Raster:   {RASTER_DIR}")
print(f"Shapes:   {SHAPES_DIR}")
print(f"Salidas:  {OUTPUT_DIR}")

Directorio configurado correctamente
Proyecto: C:\Users\Hp\Desktop\Taller_Guatemala
Raster:   C:\Users\Hp\Desktop\Taller_Guatemala\data\raster
Shapes:   C:\Users\Hp\Desktop\Taller_Guatemala\data\shapes
Salidas:  C:\Users\Hp\Desktop\Taller_Guatemala\data\raster\salidas_comparacion


In [2]:

import importlib
import subprocess
import sys

# ==========================================================
# DEPENDENCIAS DEL NOTEBOOK
# ==========================================================

def verificar_modulo(nombre, paquete_pip=None):
    try:
        modulo = importlib.import_module(nombre)
        print(f"[OK] {nombre}")
        return modulo

    except ImportError:
        if paquete_pip is None:
            raise RuntimeError(
                f"No se encontró '{nombre}'. "
                "Ejecuta este notebook con un entorno Python "
                "que tenga QGIS y GDAL correctamente instalados."
            )

        print(f"[INSTALANDO] {paquete_pip}")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            paquete_pip
        ])

        importlib.invalidate_caches()
        modulo = importlib.import_module(nombre)

        print(f"[OK] {nombre}")
        return modulo


# Librería Python instalable mediante pip
np = verificar_modulo("numpy", "numpy")

# Dependencias propias del entorno QGIS
qgis_core = verificar_modulo("qgis.core")
gdal_module = verificar_modulo("osgeo.gdal")
processing = verificar_modulo("processing")

# Comprobación de PyQt integrado en QGIS
qgis_qt = verificar_modulo("qgis.PyQt.QtGui")

print("\nTodas las dependencias están disponibles.")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"GDAL: {gdal_module.VersionInfo('--version')}")

[OK] numpy
[OK] qgis.core
[OK] osgeo.gdal
[OK] processing
[OK] qgis.PyQt.QtGui

Todas las dependencias están disponibles.
Python: 3.12.11
NumPy: 1.26.4
GDAL: GDAL 3.11.4 "Eganville", released 2025/09/04


In [3]:

import csv
import numpy as np

from osgeo import gdal
from qgis.PyQt.QtGui import QColor
from qgis.core import (
    QgsPalettedRasterRenderer,
    QgsProject,
    QgsRasterLayer,
    QgsVectorLayer
)
from processing.core.Processing import Processing

# ==========================================================
# 1. INICIALIZAR QGIS
# ==========================================================

Processing.initialize()
project = QgsProject.instance()

# ==========================================================
# 2. DEFINIR ARCHIVOS DE ENTRADA
# ==========================================================

LANDCOVER_FILENAME = "landcover_guatemala_2020_20m.tif"
STUDY_AREA_FILENAME = "03_Limites_departamentales.geojson"

CRCLAC_PATH = RASTER_DIR / "crclac-2020" / "mosaic.vrt"

local_matches = list(
    RASTER_DIR.rglob(LANDCOVER_FILENAME)
)

study_area_matches = list(
    SHAPES_DIR.rglob(STUDY_AREA_FILENAME)
)

if not CRCLAC_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontró el mosaico CRCLAC:\n{CRCLAC_PATH}"
    )

if len(local_matches) != 1:
    raise RuntimeError(
        f"Se esperaba un archivo {LANDCOVER_FILENAME}, "
        f"pero se encontraron {len(local_matches)}."
    )

if len(study_area_matches) != 1:
    raise RuntimeError(
        f"Se esperaba un archivo {STUDY_AREA_FILENAME}, "
        f"pero se encontraron {len(study_area_matches)}."
    )

LOCAL_PATH = local_matches[0]
STUDY_AREA_PATH = study_area_matches[0]

CRCLAC_LEGEND_PATH = CRCLAC_PATH.parent / "legend.csv"
LOCAL_LEGEND_PATH = LOCAL_PATH.parent / "legend.csv"

# ==========================================================
# 3. CARGAR CAPAS
# ==========================================================

crclac = QgsRasterLayer(
    str(CRCLAC_PATH),
    "CRCLAC 2020",
    "gdal"
)

local = QgsRasterLayer(
    str(LOCAL_PATH),
    "Landcover Guatemala 2020 (20 m)",
    "gdal"
)

area_estudio = QgsVectorLayer(
    str(STUDY_AREA_PATH),
    "Área de estudio - departamentos",
    "ogr"
)

for layer in [crclac, local, area_estudio]:
    if not layer.isValid():
        raise RuntimeError(
            f"No se pudo cargar la capa: {layer.name()}"
        )

    # Eliminar versiones previas para evitar duplicados
    for previous in project.mapLayersByName(layer.name()):
        project.removeMapLayer(previous.id())

    project.addMapLayer(layer)

print("[OK] Capas cargadas correctamente")

# ==========================================================
# 4. FUNCIÓN PARA LEER LEYENDAS CSV
# ==========================================================

def leer_leyenda(path, code_field, name_field, delimiter):

    if not path.is_file():
        raise FileNotFoundError(
            f"No se encontró la leyenda: {path}"
        )

    classes = []

    with path.open(
        encoding="utf-8-sig",
        newline=""
    ) as source:

        reader = csv.DictReader(
            source,
            delimiter=delimiter
        )

        if not reader.fieldnames or not {
            code_field, name_field
        }.issubset(reader.fieldnames):
            raise ValueError(
                f"La leyenda debe contener las columnas "
                f"{code_field} y {name_field}: {path}"
            )

        for index, row in enumerate(reader):

            raw_code = (row.get(code_field) or "").strip()

            if not raw_code:
                continue

            code = int(float(raw_code))

            label = " ".join(
                (row.get(name_field) or "").split()
            )

            label = "".join(
                character for character in label
                if character.isprintable()
            )

            color = QColor.fromHsv(
                int(index * 137.508) % 360,
                170,
                235
            )

            classes.append(
                QgsPalettedRasterRenderer.Class(
                    code,
                    color,
                    f"{code} - {label or 'Sin descripción'}"
                )
            )

    if not classes:
        raise ValueError(
            f"La leyenda no contiene clases válidas: {path}"
        )

    return classes

# ==========================================================
# 5. IDENTIFICAR VALORES PRESENTES EN EL RÁSTER
# ==========================================================

def valores_presentes_raster(layer, filas_por_bloque=256):

    source_path = layer.source().split("|", 1)[0]

    dataset = gdal.Open(
        source_path,
        gdal.GA_ReadOnly
    )

    if dataset is None:
        raise RuntimeError(
            f"GDAL no pudo abrir: {source_path}"
        )

    band = dataset.GetRasterBand(1)
    nodata = band.GetNoDataValue()
    present = set()

    for y_offset in range(
        0,
        dataset.RasterYSize,
        filas_por_bloque
    ):

        rows = min(
            filas_por_bloque,
            dataset.RasterYSize - y_offset
        )

        block = band.ReadAsArray(
            0,
            y_offset,
            dataset.RasterXSize,
            rows
        )

        if block is None:
            raise RuntimeError(
                f"Error leyendo bloque en fila {y_offset}"
            )

        values = np.unique(block)

        values = values[np.isfinite(values)]

        if nodata is not None:
            if np.isnan(nodata):
                values = values[~np.isnan(values)]
            else:
                values = values[values != nodata]

        present.update(
            int(value) for value in values
        )

    dataset = None

    if not present:
        raise ValueError(
            f"No hay clases válidas en {source_path}"
        )

    return present

# ==========================================================
# 6. APLICAR SIMBOLOGÍA
# ==========================================================

def aplicar_leyenda(layer, classes, solo_presentes=False):

    if solo_presentes:

        present_values = valores_presentes_raster(layer)

        classes = [
            item for item in classes
            if int(item.value) in present_values
        ]

        if not classes:
            raise ValueError(
                f"Ninguna clase coincide con {layer.name()}"
            )

    renderer = QgsPalettedRasterRenderer(
        layer.dataProvider(),
        1,
        classes
    )

    layer.setRenderer(renderer)
    layer.triggerRepaint()

    return classes

# ==========================================================
# 7. LEER Y APLICAR LEYENDAS
# ==========================================================

crclac_legend = leer_leyenda(
    CRCLAC_LEGEND_PATH,
    "Value",
    "Name (ES)",
    ","
)

local_legend = leer_leyenda(
    LOCAL_LEGEND_PATH,
    "C_USO",
    "USOT",
    ";"
)

aplicar_leyenda(crclac, crclac_legend)
aplicar_leyenda(local, local_legend)

print("[OK] Leyendas aplicadas correctamente")

# ==========================================================
# 8. INFORMACIÓN ESPACIAL
# ==========================================================

print("\n========== INFORMACIÓN ESPACIAL ==========")

print(
    "CRCLAC:",
    crclac.crs().authid(),
    "| Resolución:",
    crclac.rasterUnitsPerPixelX(),
    crclac.crs().mapUnits()
)

print(
    "Local:",
    local.crs().authid(),
    "| Resolución:",
    local.rasterUnitsPerPixelX(),
    local.crs().mapUnits()
)

print(
    "¿Mismo CRS?",
    crclac.crs() == local.crs()
)

print(
    "CRS del GeoJSON:",
    area_estudio.crs().authid()
)

print(
    "Extensión del área de estudio:",
    area_estudio.extent().toString()
)

print(
    "\nCRCLAC utiliza un mosaico virtual (VRT) "
    "que referencia los archivos de "
    "data/raster/crclac-2020/img."
)

print("\n[OK] Configuración inicial finalizada")

[OK] Capas cargadas correctamente
[OK] Leyendas aplicadas correctamente

========== INFORMACIÓN ESPACIAL ==========
CRCLAC: EPSG:4326 | Resolución: 0.00017966305682390427 6
Local: EPSG:32615 | Resolución: 20.0 0
¿Mismo CRS? False
CRS del GeoJSON: EPSG:4326
Extensión del área de estudio: -92.2401945099999949,13.7385956799999995 : -88.2215753499999948,17.8175865500000015

CRCLAC utiliza un mosaico virtual (VRT) que referencia los archivos de data/raster/crclac-2020/img.

[OK] Configuración inicial finalizada


## 2. Recortar CRCLAC con el GeoJSON del área de estudio

El recorte sigue los polígonos del GeoJSON departamental cargado en el paso 1, no solo su rectángulo envolvente. Se conserva el CRS y la resolución originales de CRCLAC; los píxeles fuera de los polígonos quedan como NoData.

In [4]:

from pathlib import Path
import processing

from qgis.core import QgsRasterLayer

# ==========================================================
# 2. RECORTAR CRCLAC AL ÁREA DE ESTUDIO
# ==========================================================

def preparar_salida(output):
    output = Path(output)
    output_resolved = output.resolve()

    # Eliminar capas que utilicen la salida anterior
    for layer in list(project.mapLayers().values()):
        source_path = layer.source().split("|", 1)[0]

        if Path(source_path).resolve() == output_resolved:
            project.removeMapLayer(layer.id())

    # Eliminar archivos de ejecuciones anteriores
    for suffix in ("", ".aux.xml", ".ovr"):
        Path(f"{output}{suffix}").unlink(missing_ok=True)


# ----------------------------------------------------------
# 1. Definir archivo de salida
# ----------------------------------------------------------

CRCLAC_CLIPPED_PATH = (
    OUTPUT_DIR / "crclac_2020_area_estudio.tif"
)

# ----------------------------------------------------------
# 2. Validar capas de entrada
# ----------------------------------------------------------

if not crclac.isValid():
    raise RuntimeError("La capa CRCLAC no es válida.")

if not area_estudio.isValid():
    raise RuntimeError("El área de estudio no es válida.")

print("CRCLAC:", crclac.source())
print("Máscara:", area_estudio.source())
print("Polígonos:", area_estudio.featureCount())

# ----------------------------------------------------------
# 3. Preparar salida
# ----------------------------------------------------------

preparar_salida(CRCLAC_CLIPPED_PATH)

# ----------------------------------------------------------
# 4. Ejecutar recorte mediante GDAL
# ----------------------------------------------------------

parametros = {
    "INPUT": crclac,
    "MASK": area_estudio,
    "SOURCE_CRS": None,
    "TARGET_CRS": crclac.crs(),
    "CROP_TO_CUTLINE": True,
    "KEEP_RESOLUTION": True,
    "MULTITHREADING": True,
    "OPTIONS": "COMPRESS=LZW|TILED=YES",
    "DATA_TYPE": 0,
    "OUTPUT": str(CRCLAC_CLIPPED_PATH),
}

print("\nEjecutando recorte de CRCLAC...")

resultado = processing.run(
    "gdal:cliprasterbymasklayer",
    parametros
)

print("Resultado del procesamiento:", resultado)

# ----------------------------------------------------------
# 5. Cargar ráster recortado
# ----------------------------------------------------------

crclac_recortado = QgsRasterLayer(
    str(CRCLAC_CLIPPED_PATH),
    "CRCLAC 2020 recortado al área de estudio",
    "gdal"
)

if not crclac_recortado.isValid():
    raise RuntimeError(
        "No se pudo cargar el ráster recortado."
    )

# Evitar capas duplicadas
for old_layer in project.mapLayersByName(
    crclac_recortado.name()
):
    project.removeMapLayer(old_layer.id())

project.addMapLayer(crclac_recortado)

# Aplicar leyenda original
aplicar_leyenda(
    crclac_recortado,
    crclac_legend
)

# ----------------------------------------------------------
# 6. Mostrar resultados
# ----------------------------------------------------------

print("\n=== RECORTE FINALIZADO ===")

print(
    "CRCLAC original:",
    crclac.width(),
    "x",
    crclac.height()
)

print(
    "CRCLAC recortado:",
    crclac_recortado.width(),
    "x",
    crclac_recortado.height()
)

print(
    "CRS:",
    crclac_recortado.crs().authid()
)

print(
    "Extensión:",
    crclac_recortado.extent().toString()
)

print(
    "Archivo generado:",
    CRCLAC_CLIPPED_PATH
)

CRCLAC: C:\Users\Hp\Desktop\Taller_Guatemala\data\raster\crclac-2020\mosaic.vrt
Máscara: C:\Users\Hp\Desktop\Taller_Guatemala\data\shapes\limites_guatemala\03_Limites_departamentales.geojson
Polígonos: 22

Ejecutando recorte de CRCLAC...
Resultado del procesamiento: {'OUTPUT': 'C:\\Users\\Hp\\Desktop\\Taller_Guatemala\\data\\raster\\salidas_comparacion\\crclac_2020_area_estudio.tif'}

=== RECORTE FINALIZADO ===
CRCLAC original: 25946 x 25698
CRCLAC recortado: 22368 x 22705
CRS: EPSG:4326
Extensión: -92.2402710147902241,13.7384746292102804 : -88.2215677597531283,17.8177243343970169
Archivo generado: C:\Users\Hp\Desktop\Taller_Guatemala\data\raster\salidas_comparacion\crclac_2020_area_estudio.tif


## 3. Filtrar las clases de cultivos

Las capas resultantes son máscaras Byte: `1` para cultivos, `0` para otras clases y NoData fuera de cobertura. Se usa el CRCLAC ya recortado en el paso anterior. Para la capa local se consideran los códigos que empiezan por `2`, excluyendo `231` (pasto cultivado) y `232` (pasto natural).

In [5]:

import processing

from pathlib import Path
from osgeo import gdal
from qgis.PyQt.QtGui import QColor
from qgis.core import (
    QgsApplication,
    QgsRasterLayer,
    QgsPalettedRasterRenderer
)

# ==========================================================
# 3. IDENTIFICAR CULTIVOS EN AMBOS PRODUCTOS
# ==========================================================

CRCLAC_CROPS_PATH = OUTPUT_DIR / "crclac_cultivos.tif"
LOCAL_CROPS_PATH = OUTPUT_DIR / "local_cultivos.tif"

# ----------------------------------------------------------
# 1. Verificar el algoritmo GDAL disponible
# ----------------------------------------------------------

algorithm = QgsApplication.processingRegistry().algorithmById(
    "gdal:rastercalculator"
)

if algorithm is None:
    raise RuntimeError(
        "No está disponible gdal:rastercalculator."
    )

parameter_names = {
    parameter.name()
    for parameter in algorithm.parameterDefinitions()
}

if "CREATION_OPTIONS" in parameter_names:
    creation_key = "CREATION_OPTIONS"
elif "OPTIONS" in parameter_names:
    creation_key = "OPTIONS"
else:
    raise RuntimeError(
        "No se encontró el parámetro de compresión GDAL."
    )

print("Parámetro de compresión:", creation_key)

# ----------------------------------------------------------
# 2. Función para crear máscaras binarias
# ----------------------------------------------------------

def calcular_raster(
    input_a,
    formula,
    output,
    input_b=None,
    projwin=None,
    output_type=0
):
    preparar_salida(output)

    parameters = {
        "INPUT_A": str(input_a),
        "BAND_A": 1,
        "FORMULA": formula,
        "NO_DATA": 255,
        "EXTENT_OPT": 0,
        "RTYPE": output_type,
        creation_key: "COMPRESS=LZW|TILED=YES",
        "OUTPUT": str(output),
    }

    if input_b is not None:
        parameters.update({
            "INPUT_B": str(input_b),
            "BAND_B": 1
        })

    if projwin is not None:
        if hasattr(projwin, "xMinimum"):
            parameters["PROJWIN"] = (
                f"{projwin.xMinimum()},"
                f"{projwin.xMaximum()},"
                f"{projwin.yMinimum()},"
                f"{projwin.yMaximum()}"
            )
        else:
            parameters["PROJWIN"] = str(projwin)

    result = processing.run(
        "gdal:rastercalculator",
        parameters
    )

    output_path = Path(result["OUTPUT"])

    if not output_path.is_file():
        raise RuntimeError(
            f"No se generó el archivo: {output_path}"
        )

    return output_path

# ----------------------------------------------------------
# 3. Crear máscara de cultivos CRCLAC
# ----------------------------------------------------------

print("\n[1/2] Identificando cultivos CRCLAC...")

calcular_raster(
    CRCLAC_CLIPPED_PATH,
    "where(A == 8192, 1, 0)",
    CRCLAC_CROPS_PATH
)

# ----------------------------------------------------------
# 4. Crear máscara de cultivos local
# ----------------------------------------------------------

print("\n[2/2] Identificando cultivos locales...")

LOCAL_CROP_RANGES = (
    (200, 300),
    (2000, 3000),
    (20000, 30000)
)

LOCAL_CROP_FORMULA = " | ".join(
    f"((A >= {lower}) & (A < {upper}))"
    for lower, upper in LOCAL_CROP_RANGES
)

LOCAL_CROP_FORMULA = (
    f"({LOCAL_CROP_FORMULA})"
    " & (A != 231)"
    " & (A != 232)"
)

calcular_raster(
    LOCAL_PATH,
    f"where({LOCAL_CROP_FORMULA}, 1, 0)",
    LOCAL_CROPS_PATH
)

# ----------------------------------------------------------
# 5. Función para cargar resultados en QGIS
# ----------------------------------------------------------

def agregar_raster(path, name, classes=None):

    layer = QgsRasterLayer(str(path), name, "gdal")

    if not layer.isValid():
        raise RuntimeError(
            f"No se pudo cargar el ráster: {path}"
        )

    for old_layer in project.mapLayersByName(name):
        project.removeMapLayer(old_layer.id())

    project.addMapLayer(layer)

    if classes is not None:
        palette = []

        for item in classes:
            if isinstance(
                item, QgsPalettedRasterRenderer.Class
            ):
                palette.append(item)
            else:
                value, color, label = item
                palette.append(
                    QgsPalettedRasterRenderer.Class(
                        value,
                        QColor(color),
                        label
                    )
                )

        aplicar_leyenda(layer, palette)

    return layer

# ----------------------------------------------------------
# 6. Aplicar simbología
# ----------------------------------------------------------

crop_style = [
    (0, QColor(255, 255, 255, 0), "Otras clases"),
    (1, "#31a354", "Cultivos")
]

crclac_cultivos = agregar_raster(
    CRCLAC_CROPS_PATH,
    "Cultivos CRCLAC 2020",
    crop_style
)

local_cultivos = agregar_raster(
    LOCAL_CROPS_PATH,
    "Cultivos landcover local 2020",
    crop_style
)

# ----------------------------------------------------------
# 7. Validar resultados
# ----------------------------------------------------------

print("\n=== RESULTADOS: MÁSCARAS DE CULTIVOS ===")

for name, path in [
    ("CRCLAC", CRCLAC_CROPS_PATH),
    ("Local", LOCAL_CROPS_PATH)
]:
    ds = gdal.Open(str(path), gdal.GA_ReadOnly)

    if ds is None:
        raise RuntimeError(f"No se pudo abrir {path}")

    band = ds.GetRasterBand(1)

    print(f"\n{name}")
    print("Archivo:", path.name)
    print("Dimensiones:", ds.RasterXSize, "x", ds.RasterYSize)
    print("Tipo de dato:", gdal.GetDataTypeName(band.DataType))
    print("NoData:", band.GetNoDataValue())

    if band.DataType != gdal.GDT_Byte:
        raise RuntimeError(
            f"La máscara {name} no tiene tipo Byte."
        )

    if band.GetNoDataValue() != 255:
        raise RuntimeError(
            f"La máscara {name} no tiene NoData=255."
        )

    ds = None

print("\n[OK] Ambas máscaras fueron creadas correctamente.")

Parámetro de compresión: OPTIONS

[1/2] Identificando cultivos CRCLAC...

[2/2] Identificando cultivos locales...

=== RESULTADOS: MÁSCARAS DE CULTIVOS ===

CRCLAC
Archivo: crclac_cultivos.tif
Dimensiones: 22368 x 22705
Tipo de dato: Byte
NoData: 255.0

Local
Archivo: local_cultivos.tif
Dimensiones: 21532 x 22665
Tipo de dato: Byte
NoData: 255.0

[OK] Ambas máscaras fueron creadas correctamente.


## 4. Alinear la clasificación original de CRCLAC a la grilla local

Se reproyectan los códigos originales de CRCLAC (no la máscara binaria) al CRS y grilla local, con vecino más cercano. Así el cruce conserva qué clase CRCLAC corresponde a cada píxel y permite construir la matriz de clases.

In [6]:

from pathlib import Path
from osgeo import gdal
from qgis.core import QgsRasterLayer

# ==========================================================
# 4. ALINEAR CRCLAC A LA GRILLA DEL PRODUCTO LOCAL
# ==========================================================

CRCLAC_ALIGNED_PATH = (
    OUTPUT_DIR / "crclac_clases_alineadas_a_local.tif"
)

# ----------------------------------------------------------
# 1. Abrir archivos de referencia
# ----------------------------------------------------------

src_ds = gdal.Open(
    str(CRCLAC_CLIPPED_PATH),
    gdal.GA_ReadOnly
)

ref_ds = gdal.Open(
    str(LOCAL_PATH),
    gdal.GA_ReadOnly
)

if src_ds is None or ref_ds is None:
    raise RuntimeError(
        "No se pudieron abrir los raster de entrada."
    )

# ----------------------------------------------------------
# 2. Obtener características de la grilla local
# ----------------------------------------------------------

gt = ref_ds.GetGeoTransform()

cols = ref_ds.RasterXSize
rows = ref_ds.RasterYSize

if abs(gt[2]) > 1e-12 or abs(gt[4]) > 1e-12:
    raise RuntimeError(
        "La grilla local presenta rotación o inclinación."
    )

xmin = gt[0]
ymax = gt[3]

xmax = xmin + cols * gt[1]
ymin = ymax + rows * gt[5]

bounds = (
    min(xmin, xmax),
    min(ymin, ymax),
    max(xmin, xmax),
    max(ymin, ymax)
)

print("=== GRILLA DE REFERENCIA ===")
print("Columnas:", cols)
print("Filas:", rows)
print("Resolución X:", gt[1])
print("Resolución Y:", gt[5])
print("Extensión:", bounds)
print("CRS local:", local.crs().authid())

# ----------------------------------------------------------
# 3. Identificar NoData del producto original
# ----------------------------------------------------------

src_band = src_ds.GetRasterBand(1)

src_nodata = src_band.GetNoDataValue()
src_type = src_band.DataType

# Utilizamos 0 como NoData de destino.
# Verificar que no sea un código temático válido.
codigos_crclac = {
    int(item.value)
    for item in crclac_legend
}

if 0 in codigos_crclac:
    raise RuntimeError(
        "El código 0 aparece en la leyenda CRCLAC. "
        "Se debe elegir otro valor NoData antes de continuar."
    )

dst_nodata = 0

# Cerrar datasets antes de generar la salida
src_ds = None
ref_ds = None

# ----------------------------------------------------------
# 4. Ejecutar alineación mediante GDAL Warp
# ----------------------------------------------------------

preparar_salida(CRCLAC_ALIGNED_PATH)

print("\nAlineando CRCLAC a la grilla local...")

options = gdal.WarpOptions(
    format="GTiff",
    dstSRS=local.crs().toWkt(),
    outputBounds=bounds,
    width=cols,
    height=rows,
    resampleAlg=gdal.GRA_NearestNeighbour,
    srcNodata=src_nodata,
    dstNodata=dst_nodata,
    outputType=src_type,
    multithread=True,
    creationOptions=[
        "COMPRESS=LZW",
        "TILED=YES",
        "BIGTIFF=IF_SAFER"
    ]
)

aligned_ds = gdal.Warp(
    str(CRCLAC_ALIGNED_PATH),
    str(CRCLAC_CLIPPED_PATH),
    options=options
)

if aligned_ds is None:
    raise RuntimeError(
        "GDAL no pudo generar el raster alineado."
    )

aligned_ds.FlushCache()
aligned_ds = None

# ----------------------------------------------------------
# 5. Verificar alineación espacial
# ----------------------------------------------------------

ref_ds = gdal.Open(str(LOCAL_PATH))
aligned_ds = gdal.Open(str(CRCLAC_ALIGNED_PATH))

if aligned_ds is None:
    raise RuntimeError(
        "No se pudo abrir el resultado alineado."
    )

same_size = (
    ref_ds.RasterXSize == aligned_ds.RasterXSize
    and ref_ds.RasterYSize == aligned_ds.RasterYSize
)

same_transform = all(
    abs(a - b) < 1e-9
    for a, b in zip(
        ref_ds.GetGeoTransform(),
        aligned_ds.GetGeoTransform()
    )
)

same_crs = bool(
    ref_ds.GetSpatialRef()
    and aligned_ds.GetSpatialRef()
    and ref_ds.GetSpatialRef().IsSame(
        aligned_ds.GetSpatialRef()
    )
)

print("\n=== VALIDACIÓN DE ALINEACIÓN ===")
print("Mismas dimensiones:", same_size)
print("Mismo geotransform:", same_transform)
print("Mismo CRS:", same_crs)

if not all([same_size, same_transform, same_crs]):
    raise RuntimeError(
        "La alineación espacial requiere revisión."
    )

print(
    "Dimensiones:",
    aligned_ds.RasterXSize,
    "x",
    aligned_ds.RasterYSize
)

print(
    "NoData:",
    aligned_ds.GetRasterBand(1).GetNoDataValue()
)

ref_ds = None
aligned_ds = None

# ----------------------------------------------------------
# 6. Cargar resultado en QGIS
# ----------------------------------------------------------

crclac_alineado = agregar_raster(
    CRCLAC_ALIGNED_PATH,
    "CRCLAC 2020 alineado a grilla local",
    crclac_legend
)

print("\n[OK] Alineación completada correctamente.")
print("Archivo generado:", CRCLAC_ALIGNED_PATH)

=== GRILLA DE REFERENCIA ===
Columnas: 21532
Filas: 22665
Resolución X: 20.0
Resolución Y: -20.0
Extensión: (581840.0, 1520760.0, 1012480.0, 1974060.0)
CRS local: EPSG:32615

Alineando CRCLAC a la grilla local...

=== VALIDACIÓN DE ALINEACIÓN ===
Mismas dimensiones: True
Mismo geotransform: True
Mismo CRS: True
Dimensiones: 21532 x 22665
NoData: 0.0

[OK] Alineación completada correctamente.
Archivo generado: C:\Users\Hp\Desktop\Taller_Guatemala\data\raster\salidas_comparacion\crclac_clases_alineadas_a_local.tif


## 5. Cruce de las dos clasificaciones

El cruce conserva cuatro estados: `0` = ninguno clasifica como cultivo, `1` = ambos, `2` = sólo CRCLAC y `3` = sólo local. Se basa en las clases originales alineadas de CRCLAC y la máscara agrícola local. El código `3` selecciona los píxeles para la matriz de clases del paso siguiente.

In [7]:

from pathlib import Path
import numpy as np
from osgeo import gdal

# ==========================================================
# 5. CRUCE DE CULTIVOS: CRCLAC VS PRODUCTO LOCAL
# ==========================================================

CROSS_PATH = (
    OUTPUT_DIR / "cruce_cultivos_crclac_vs_local.tif"
)

# 1. Abrir los raster de entrada
cr_ds = gdal.Open(
    str(CRCLAC_ALIGNED_PATH), gdal.GA_ReadOnly
)

loc_ds = gdal.Open(
    str(LOCAL_CROPS_PATH), gdal.GA_ReadOnly
)

if cr_ds is None or loc_ds is None:
    raise RuntimeError(
        "No se pudieron abrir los raster de entrada."
    )

# 2. Verificar coincidencia exacta de grillas
if (
    cr_ds.RasterXSize != loc_ds.RasterXSize
    or cr_ds.RasterYSize != loc_ds.RasterYSize
):
    raise RuntimeError(
        "Los raster tienen dimensiones distintas."
    )

if not np.allclose(
    cr_ds.GetGeoTransform(),
    loc_ds.GetGeoTransform(),
    rtol=0,
    atol=1e-9
):
    raise RuntimeError(
        "Los raster no tienen el mismo geotransform."
    )

if not (
    cr_ds.GetSpatialRef()
    and loc_ds.GetSpatialRef()
    and cr_ds.GetSpatialRef().IsSame(
        loc_ds.GetSpatialRef()
    )
):
    raise RuntimeError(
        "Los raster no tienen el mismo CRS."
    )

# 3. Preparar salida
preparar_salida(CROSS_PATH)

driver = gdal.GetDriverByName("GTiff")

out_ds = driver.Create(
    str(CROSS_PATH),
    cr_ds.RasterXSize,
    cr_ds.RasterYSize,
    1,
    gdal.GDT_Byte,
    options=[
        "COMPRESS=LZW",
        "TILED=YES",
        "BIGTIFF=IF_SAFER"
    ]
)

if out_ds is None:
    raise RuntimeError("No se pudo crear el raster de cruce.")

out_ds.SetGeoTransform(cr_ds.GetGeoTransform())
out_ds.SetProjection(cr_ds.GetProjection())

out_band = out_ds.GetRasterBand(1)
out_band.SetNoDataValue(255)

cr_band = cr_ds.GetRasterBand(1)
loc_band = loc_ds.GetRasterBand(1)

cr_nodata = cr_band.GetNoDataValue()
loc_nodata = loc_band.GetNoDataValue()

conteos = np.zeros(4, dtype=np.int64)
total_nodata = 0

# 4. Procesar por bloques para controlar memoria
bloque = 512

for y in range(0, cr_ds.RasterYSize, bloque):

    nrows = min(bloque, cr_ds.RasterYSize - y)

    a = cr_band.ReadAsArray(
        0, y, cr_ds.RasterXSize, nrows
    )

    b = loc_band.ReadAsArray(
        0, y, loc_ds.RasterXSize, nrows
    )

    if a is None or b is None:
        raise RuntimeError(
            f"Error leyendo el bloque en fila {y}"
        )

    valid = np.isfinite(a) & np.isfinite(b)

    if cr_nodata is not None:
        valid &= a != cr_nodata

    if loc_nodata is not None:
        valid &= b != loc_nodata

    # Clasificación de coincidencias
    cultivo_cr = a == 8192
    cultivo_local = b == 1

    resultado = np.full(a.shape, 255, dtype=np.uint8)

    resultado[valid] = (
        cultivo_cr[valid].astype(np.uint8) * 2
        + cultivo_local[valid].astype(np.uint8)
    )

    # Convertir códigos:
    # 0 ninguno, 1 ambos, 2 solo CRCLAC, 3 solo local
    resultado[valid & cultivo_cr & cultivo_local] = 1
    resultado[valid & cultivo_cr & ~cultivo_local] = 2
    resultado[valid & ~cultivo_cr & cultivo_local] = 3

    out_band.WriteArray(resultado, 0, y)

    for codigo in range(4):
        conteos[codigo] += np.count_nonzero(
            resultado == codigo
        )

    total_nodata += np.count_nonzero(resultado == 255)

out_band.FlushCache()

out_ds = None
cr_ds = None
loc_ds = None

# 5. Cargar el resultado en QGIS
cross_style = [
    (0, "#d9d9d9", "Ninguno clasifica como cultivo"),
    (1, "#238b45", "Cultivo en ambos"),
    (2, "#fdae6b", "Sólo CRCLAC"),
    (3, "#6baed6", "Sólo local")
]

cruce = agregar_raster(
    CROSS_PATH,
    "Cruce de cultivos: CRCLAC vs local",
    cross_style
)

# 6. Calcular superficie por categoría
# Grilla UTM de 20 x 20 m = 400 m2 = 0,04 ha
area_pixel_ha = 20 * 20 / 10000

categorias = [
    "Ninguno",
    "Ambos",
    "Solo CRCLAC",
    "Solo local"
]

print("\n=== RESULTADOS DEL CRUCE ===")

for codigo, nombre in enumerate(categorias):
    pixeles = int(conteos[codigo])
    hectareas = pixeles * area_pixel_ha

    print(
        f"{codigo} - {nombre}: "
        f"{pixeles:,} pixeles | "
        f"{hectareas:,.2f} ha"
    )

print("NoData:", f"{total_nodata:,}", "pixeles")

print("\n[OK] Cruce generado correctamente.")
print("Archivo:", CROSS_PATH)


=== RESULTADOS DEL CRUCE ===
0 - Ninguno: 166,108,331 pixeles | 6,644,333.24 ha
1 - Ambos: 35,527,980 pixeles | 1,421,119.20 ha
2 - Solo CRCLAC: 27,103,189 pixeles | 1,084,127.56 ha
3 - Solo local: 41,545,610 pixeles | 1,661,824.40 ha
NoData: 217,737,670 pixeles

[OK] Cruce generado correctamente.
Archivo: C:\Users\Hp\Desktop\Taller_Guatemala\data\raster\salidas_comparacion\cruce_cultivos_crclac_vs_local.tif


## 6. Matriz de clases locales no mapeadas y clases originales de CRCLAC

La capa `Cruce de cultivos: CRCLAC vs local` se usa como máscara: solo se cuentan píxeles con código `3` (`Sólo local`). Para esos píxeles, se cruza el código local de cultivo con el código original de CRCLAC ya alineado. El resultado muestra en qué clases de CRCLAC cae cada cultivo local no identificado como `8192`. El análisis cuenta todos los píxeles por bloques y excluye NoData.

La matriz resultante está condicionada a la categoría `Sólo local`; no es una matriz de exactitud contra verdad de terreno ni incluye los píxeles coincidentes.

In [8]:

# ==========================================================
# EJERCICIO 6 — MATRIZ DE CLASES Y DISCREPANCIAS
# CRCLAC 2020 vs. LAND COVER GUATEMALA 2020
# ==========================================================

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from osgeo import gdal

# ----------------------------------------------------------
# 1. CONFIGURAR RUTAS
# ----------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_DIR / "data" / "raster" / "salidas_comparacion"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CROSS_PATH = (
    OUTPUT_DIR / "cruce_cultivos_crclac_vs_local.tif"
)

CRCLAC_ALIGNED_PATH = (
    OUTPUT_DIR / "crclac_clases_alineadas_a_local.tif"
)

# LOCAL_PATH se definió en la celda de carga de datos

for ruta in [CROSS_PATH, CRCLAC_ALIGNED_PATH, LOCAL_PATH]:
    if not Path(ruta).is_file():
        raise FileNotFoundError(
            f"Falta un archivo del procesamiento:\n{ruta}\n"
            "Ejecuta los ejercicios anteriores."
        )

# ----------------------------------------------------------
# 2. ABRIR LOS RASTER
# ----------------------------------------------------------

cross_ds = gdal.Open(str(CROSS_PATH))
local_ds = gdal.Open(str(LOCAL_PATH))
crclac_ds = gdal.Open(str(CRCLAC_ALIGNED_PATH))

if any(ds is None for ds in [cross_ds, local_ds, crclac_ds]):
    raise RuntimeError("No se pudieron abrir los raster.")

datasets = [cross_ds, local_ds, crclac_ds]

referencia = local_ds

for ds in datasets:
    if (
        ds.RasterXSize != referencia.RasterXSize
        or ds.RasterYSize != referencia.RasterYSize
        or not np.allclose(
            ds.GetGeoTransform(),
            referencia.GetGeoTransform(),
            atol=1e-9,
            rtol=0
        )
    ):
        raise RuntimeError(
            "Los raster no comparten la misma grilla."
        )

    if not (
        ds.GetSpatialRef()
        and referencia.GetSpatialRef()
        and ds.GetSpatialRef().IsSame(
            referencia.GetSpatialRef()
        )
    ):
        raise RuntimeError(
            "Los raster no comparten el mismo CRS."
        )

# ----------------------------------------------------------
# 3. RESOLUCIÓN Y SUPERFICIE POR PÍXEL
# ----------------------------------------------------------

srs = local_ds.GetSpatialRef()

if srs is None or not srs.IsProjected():
    raise RuntimeError(
        "Se requiere un CRS proyectado en metros."
    )

gt = local_ds.GetGeoTransform()

area_pixel_ha = abs(
    gt[1] * gt[5] - gt[2] * gt[4]
) / 10000

print("=== INFORMACIÓN ESPACIAL ===")
print("Resolución:", abs(gt[1]), "m")
print("Superficie por píxel:", area_pixel_ha, "ha")

# ----------------------------------------------------------
# 4. PREPARAR CONTADORES
# ----------------------------------------------------------

cross_band = cross_ds.GetRasterBand(1)
local_band = local_ds.GetRasterBand(1)
crclac_band = crclac_ds.GetRasterBand(1)

cross_nodata = cross_band.GetNoDataValue()
local_nodata = local_band.GetNoDataValue()
crclac_nodata = crclac_band.GetNoDataValue()

cross_counts = Counter()
class_counts = defaultdict(Counter)
unmapped_pairs = Counter()

# ----------------------------------------------------------
# 5. PROCESAMIENTO POR BLOQUES
# ----------------------------------------------------------

BLOCK_SIZE = 256

print("\nProcesando matriz de clases...")

for y in range(0, cross_ds.RasterYSize, BLOCK_SIZE):

    rows = min(
        BLOCK_SIZE,
        cross_ds.RasterYSize - y
    )

    a = cross_band.ReadAsArray(
        0, y, cross_ds.RasterXSize, rows
    )

    b = local_band.ReadAsArray(
        0, y, local_ds.RasterXSize, rows
    )

    c = crclac_band.ReadAsArray(
        0, y, crclac_ds.RasterXSize, rows
    )

    if any(block is None for block in [a, b, c]):
        raise RuntimeError(
            f"Error leyendo bloque en fila {y}"
        )

    valid = (
        np.isfinite(a)
        & np.isfinite(b)
        & np.isfinite(c)
    )

    for block, nodata in [
        (a, cross_nodata),
        (b, local_nodata),
        (c, crclac_nodata)
    ]:
        if nodata is not None:
            valid &= block != nodata

    # Conteo por categoría de cruce
    codes, counts = np.unique(
        a[valid],
        return_counts=True
    )

    for code, count in zip(codes, counts):
        cross_counts[int(code)] += int(count)

    # Identificar cultivos locales
    local_crop = (
        ((b >= 200) & (b < 300))
        | ((b >= 2000) & (b < 3000))
        | ((b >= 20000) & (b < 30000))
    )

    local_crop &= (b != 231) & (b != 232)

    mask = valid & local_crop

    # Contar por código local y estado de cruce
    if np.any(mask):
        triples = np.column_stack((
            b[mask],
            a[mask]
        ))

        unique, frequencies = np.unique(
            triples, axis=0, return_counts=True
        )

        for (local_code, status), count in zip(
            unique, frequencies
        ):
            if int(status) not in (1, 3):
                raise RuntimeError(
                    "Se detectaron cultivos locales "
                    "fuera de las categorías 1 y 3."
                )

            class_counts[int(local_code)][int(status)] += int(count)

    # Cruce específico para Solo local = 3
    only_local = mask & (a == 3)

    if np.any(only_local):
        pairs = np.column_stack((
            b[only_local],
            c[only_local]
        ))

        unique, frequencies = np.unique(
            pairs, axis=0, return_counts=True
        )

        for (local_code, cr_code), count in zip(
            unique, frequencies
        ):
            unmapped_pairs[
                (int(local_code), int(cr_code))
            ] += int(count)

cross_ds = None
local_ds = None
crclac_ds = None

# ----------------------------------------------------------
# 6. MATRIZ GENERAL DE ACUERDO
# ----------------------------------------------------------

confusion = pd.DataFrame(
    [
        [cross_counts[1], cross_counts[3]],
        [cross_counts[2], cross_counts[0]]
    ],
    index=[
        "Local: cultivos",
        "Local: otras clases"
    ],
    columns=[
        "CRCLAC: cultivos",
        "CRCLAC: otras clases"
    ]
)

print("\n=== MATRIZ DE ACUERDO ===")
print(confusion.to_string())

print(
    "\nPíxeles válidos:",
    int(confusion.to_numpy().sum())
)

# ----------------------------------------------------------
# 7. LEYENDAS
# ----------------------------------------------------------

local_names = {
    int(item.value): item.label
    for item in local_legend
}

crclac_names = {
    int(item.value): item.label
    for item in crclac_legend
}

def etiqueta_clase(code, names):
    nombre = str(names.get(code, code))
    prefijo = f"{code} - "

    if nombre.startswith(prefijo):
        return nombre

    return prefijo + nombre

# ----------------------------------------------------------
# 8. RESUMEN POR CLASE LOCAL
# ----------------------------------------------------------

rows = []

for code, counts in sorted(class_counts.items()):

    ambos = counts[1]
    solo_local = counts[3]
    total = ambos + solo_local

    if total == 0:
        continue

    rows.append({
        "codigo": code,
        "clase_local": etiqueta_clase(
            code, local_names
        ),
        "ambos_pixeles": ambos,
        "solo_local_pixeles": solo_local,
        "cultivo_local_pixeles": total,
        "solo_local_pct": 100 * solo_local / total,
        "ambos_ha": ambos * area_pixel_ha,
        "solo_local_ha": solo_local * area_pixel_ha
    })

if not rows:
    raise RuntimeError(
        "No se encontraron clases agrícolas válidas."
    )

class_summary = pd.DataFrame(rows).sort_values(
    "solo_local_ha", ascending=False
)

CSV_SUMMARY_PATH = (
    OUTPUT_DIR / "resumen_clases_locales_no_mapeadas.csv"
)

class_summary.to_csv(
    CSV_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ----------------------------------------------------------
# 9. DETALLE DE PARES DE CLASES
# ----------------------------------------------------------

pair_rows = []

for (local_code, cr_code), count in unmapped_pairs.items():

    total_local = class_counts[local_code][3]

    pair_rows.append({
        "codigo_local": local_code,
        "clase_local": etiqueta_clase(
            local_code, local_names
        ),
        "codigo_crclac": cr_code,
        "clase_crclac": etiqueta_clase(
            cr_code, crclac_names
        ),
        "pixeles": count,
        "hectareas": count * area_pixel_ha,
        "pct_de_clase_local_solo_local":
            100 * count / total_local
    })

if not pair_rows:
    raise RuntimeError(
        "No se encontraron píxeles Solo local."
    )

pair_summary = pd.DataFrame(pair_rows)

PAIR_DETAIL_PATH = (
    OUTPUT_DIR / "detalle_pares_local_vs_crclac_solo_local.csv"
)

pair_summary.to_csv(
    PAIR_DETAIL_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ----------------------------------------------------------
# 10. MATRICES DE PÍXELES Y HECTÁREAS
# ----------------------------------------------------------

matrix_pixels = pair_summary.pivot_table(
    index="clase_local",
    columns="clase_crclac",
    values="pixeles",
    aggfunc="sum",
    fill_value=0
).astype("int64")

matrix_hectares = pair_summary.pivot_table(
    index="clase_local",
    columns="clase_crclac",
    values="hectareas",
    aggfunc="sum",
    fill_value=0
)

# Ordenar por superficie Solo local
class_order = (
    class_summary.loc[
        class_summary["solo_local_pixeles"] > 0,
        "clase_local"
    ].tolist()
)

matrix_pixels = matrix_pixels.reindex(
    class_order, fill_value=0
)

matrix_hectares = matrix_hectares.reindex(
    class_order, fill_value=0
)

MATRIX_PIXELS_PATH = (
    OUTPUT_DIR
    / "matriz_clases_locales_vs_crclac_solo_local_pixeles.csv"
)

MATRIX_HECTARES_PATH = (
    OUTPUT_DIR
    / "matriz_clases_locales_vs_crclac_solo_local_hectareas.csv"
)

matrix_pixels.to_csv(
    MATRIX_PIXELS_PATH,
    encoding="utf-8-sig"
)

matrix_hectares.to_csv(
    MATRIX_HECTARES_PATH,
    encoding="utf-8-sig",
    float_format="%.2f"
)

# ----------------------------------------------------------
# 11. VALIDACIÓN DE RESULTADOS
# ----------------------------------------------------------

if set(cross_counts) - {0, 1, 2, 3}:
    raise RuntimeError(
        "Se encontraron códigos de cruce no esperados."
    )

if int(matrix_pixels.to_numpy().sum()) != cross_counts[3]:
    raise RuntimeError(
        "La matriz Solo local no coincide "
        "con el conteo del cruce."
    )

# ----------------------------------------------------------
# 12. MOSTRAR RESUMEN
# ----------------------------------------------------------

print("\n=== CLASES CON MÁS DISCREPANCIAS ===")

print(
    class_summary[
        ["clase_local", "solo_local_ha", "solo_local_pct"]
    ].head(15).to_string(index=False)
)

print("\n=== ARCHIVOS GENERADOS ===")

for archivo in [
    CSV_SUMMARY_PATH,
    MATRIX_PIXELS_PATH,
    MATRIX_HECTARES_PATH,
    PAIR_DETAIL_PATH
]:
    print("[OK]", archivo.name)

print("\n[OK] Ejercicio 6 completado correctamente.")
print("Ya puedes ejecutar la celda de figuras.")


=== INFORMACIÓN ESPACIAL ===
Resolución: 20.0 m
Superficie por píxel: 0.04 ha

Procesando matriz de clases...

=== MATRIZ DE ACUERDO ===
                     CRCLAC: cultivos  CRCLAC: otras clases
Local: cultivos              35527980              41545610
Local: otras clases          27103189             166108331

Píxeles válidos: 270285110

=== CLASES CON MÁS DISCREPANCIAS ===
                                                                 clase_local  solo_local_ha  solo_local_pct
                                        211 - Granos básicos (maíz y frijol)      645537.84       55.538809
                                                                 2221 - Café      470461.20       87.500538
                                                            2242 - Cardamomo      151707.60       89.371088
                                                                 2236 - Hule      108455.40       77.412936
                                                     2421 - Café y cardamomo 

In [9]:

# ==========================================================
# EJERCICIO 6 — VISUALIZACIÓN FINAL DE RESULTADOS
# Figuras independientes + figura combinada 1x2
# ==========================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ----------------------------------------------------------
# 1. DIRECTORIOS Y ARCHIVOS
# ----------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_DIR
    / "data"
    / "raster"
    / "salidas_comparacion"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESUMEN_PATH = (
    OUTPUT_DIR / "resumen_clases_locales_no_mapeadas.csv"
)

MATRIZ_PATH = (
    OUTPUT_DIR
    / "matriz_clases_locales_vs_crclac_solo_local_hectareas.csv"
)

for archivo in [RESUMEN_PATH, MATRIZ_PATH]:
    if not archivo.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo:\n{archivo}"
        )

# ----------------------------------------------------------
# 2. LEER RESULTADOS
# ----------------------------------------------------------

resumen = pd.read_csv(RESUMEN_PATH, encoding="utf-8-sig")

matriz = pd.read_csv(
    MATRIZ_PATH,
    index_col=0,
    encoding="utf-8-sig"
)

resumen.columns = resumen.columns.str.strip()
matriz.index = matriz.index.astype(str).str.strip()
matriz.columns = matriz.columns.astype(str).str.strip()

# ----------------------------------------------------------
# 3. NORMALIZAR CÓDIGOS
# ----------------------------------------------------------

def extraer_codigo(texto):
    """Extrae el primer código numérico de una etiqueta."""
    match = re.match(r"^\s*(\d+)", str(texto))

    if match is None:
        raise ValueError(
            f"No se pudo identificar el código: {texto}"
        )

    return int(match.group(1))


def extraer_nombre(texto):
    """Elimina códigos repetidos de una etiqueta."""
    nombre = re.sub(
        r"^\s*\d+\s*-\s*(?:\d+\s*-\s*)?",
        "",
        str(texto)
    )
    return nombre.strip()


# Códigos locales del resumen
resumen["codigo"] = (
    resumen["clase_local"].apply(extraer_codigo)
)

# Códigos locales de la matriz
matriz.index = [
    extraer_codigo(valor)
    for valor in matriz.index
]

# Consolidar por código local
matriz = matriz.groupby(level=0).sum()

# Normalizar códigos de CRCLAC
matriz.columns = [
    extraer_codigo(col)
    for col in matriz.columns
]

matriz = matriz.T.groupby(level=0).sum().T

# ----------------------------------------------------------
# 4. SELECCIONAR LAS 10 CLASES MÁS RELEVANTES
# ----------------------------------------------------------

TOP_N = 10

top = (
    resumen
    .sort_values("solo_local_ha", ascending=False)
    .drop_duplicates(subset=["codigo"])
    .head(TOP_N)
    .copy()
)

top["nombre"] = (
    top["clase_local"].apply(extraer_nombre)
)

top["etiqueta"] = (
    top["codigo"].astype(str)
    + " · "
    + top["nombre"]
)

# Comprobar correspondencia
faltantes = (
    set(top["codigo"]) - set(matriz.index)
)

if faltantes:
    raise ValueError(
        f"Clases ausentes de la matriz: {faltantes}"
    )

# Seleccionar y ordenar filas por código
matriz_top = matriz.loc[top["codigo"]].copy()

# Reordenar columnas por superficie total
orden_columnas = (
    matriz_top.sum(axis=0)
    .sort_values(ascending=False)
    .index
)

matriz_top = matriz_top[orden_columnas]

# Excluir columnas sin superficie
matriz_top = matriz_top.loc[
    :, matriz_top.sum(axis=0) > 0
]

# ----------------------------------------------------------
# 5. PALETA Y ETIQUETAS
# ----------------------------------------------------------

colores_crclac = {
    12288: "#377eb8",  # Pastizales
    16384: "#8da0cb",  # Cobertura arbórea
    20480: "#fc8d62",  # Cobertura arbustiva
    24576: "#e5c494",  # Vegetación inundada
    28672: "#66c2a5",  # Manglares
    36864: "#a6d854",  # Suelo desnudo
    4096:  "#e41a1c",  # Superficies artificiales
    40960: "#ffd92f",  # Nieve y glaciares
    45056: "#984ea3",  # Agua
}

nombres_crclac = {
    12288: "Pastizales",
    16384: "Cobertura arbórea",
    20480: "Cobertura arbustiva",
    24576: "Vegetación herbácea inundada",
    28672: "Manglares",
    36864: "Suelo desnudo",
    4096: "Superficies artificiales",
    40960: "Nieve y glaciares",
    45056: "Cuerpos de agua",
}

# ----------------------------------------------------------
# 6. MATRIZ DE ACUERDO
# ----------------------------------------------------------

acuerdo = np.array([
    [35527980, 41545610],
    [27103189, 166108331]
], dtype=np.int64)

# Porcentaje por fila
porcentajes = (
    acuerdo / acuerdo.sum(axis=1, keepdims=True) * 100
)

def dibujar_matriz(ax):

    # Paleta de cuatro categorías
    colores = np.array([
        [
            [35/255, 139/255, 69/255],
            [107/255, 174/255, 214/255]
        ],
        [
            [253/255, 174/255, 107/255],
            [217/255, 217/255, 217/255]
        ]
    ])

    ax.imshow(colores, aspect="equal")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels([
        "Cultivos",
        "Otras clases"
    ], fontsize=10)

    ax.set_yticklabels([
        "Cultivos",
        "Otras clases"
    ], fontsize=10)

    ax.set_xlabel(
        "Clasificación CRCLAC",
        fontsize=11,
        labelpad=10
    )

    ax.set_ylabel(
        "Clasificación local",
        fontsize=11,
        labelpad=10
    )

    ax.set_title(
        "A. Matriz de acuerdo",
        fontsize=14,
        fontweight="bold",
        pad=18
    )

    for i in range(2):
        for j in range(2):

            cantidad = acuerdo[i, j]
            porcentaje = porcentajes[i, j]

            ax.text(
                j, i,
                f"{cantidad:,}\n({porcentaje:.1f} %)",
                ha="center",
                va="center",
                fontsize=11,
                color="#222222",
                fontweight="medium"
            )

    ax.set_xticks(
        np.arange(-0.5, 2, 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(-0.5, 2, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=3
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

# ----------------------------------------------------------
# 7. GRÁFICO DE BARRAS APILADAS
# ----------------------------------------------------------

def dibujar_barras(ax):

    y = np.arange(len(matriz_top))
    acumulado = np.zeros(len(matriz_top))

    for codigo in matriz_top.columns:

        valores = matriz_top[codigo].to_numpy(
            dtype=float
        )

        ax.barh(
            y,
            valores,
            left=acumulado,
            color=colores_crclac.get(
                codigo, "#999999"
            ),
            height=0.72,
            edgecolor="white",
            linewidth=0.35
        )

        acumulado += valores

    ax.set_yticks(y)

    ax.set_yticklabels(
        top["etiqueta"].tolist(),
        fontsize=10
    )

    ax.invert_yaxis()

    ax.set_xlabel(
        "Superficie de discrepancia (ha)",
        fontsize=11
    )

    ax.set_title(
        "B. Clases CRCLAC donde el producto local identifica cultivos",
        fontsize=13,
        fontweight="bold",
        pad=18
    )

    ax.grid(
        axis="x",
        alpha=0.18,
        linestyle="--"
    )

    ax.set_axisbelow(True)

    for spine in [
        "top", "right", "left"
    ]:
        ax.spines[spine].set_visible(False)

    ax.tick_params(axis="y", length=0)

    # Etiquetas de superficie
    max_total = max(acumulado)

    ax.set_xlim(
        0,
        max_total * 1.19
    )

    for yi, total in enumerate(acumulado):

        etiqueta_total = (
            f"{total:,.0f}".replace(",", ".")
            + " ha"
        )

        ax.text(
            total + max_total * 0.012,
            yi,
            etiqueta_total,
            va="center",
            fontsize=9
        )

# ----------------------------------------------------------
# 8. LEYENDA COMPARTIDA
# ----------------------------------------------------------

handles = [
    Patch(
        facecolor=colores_crclac.get(
            codigo, "#999999"
        ),
        label=nombres_crclac.get(
            codigo, str(codigo)
        )
    )
    for codigo in matriz_top.columns
]

# ----------------------------------------------------------
# 9. FIGURA A — MATRIZ INDEPENDIENTE
# ----------------------------------------------------------

fig1, ax1 = plt.subplots(
    figsize=(7, 6),
    facecolor="white"
)

dibujar_matriz(ax1)

fig1.tight_layout(pad=2)

ruta_fig1 = (
    OUTPUT_DIR / "fig_acuerdo_productos.png"
)

fig1.savefig(
    ruta_fig1,
    dpi=250,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig1)

# ----------------------------------------------------------
# 10. FIGURA B — BARRAS INDEPENDIENTES
# ----------------------------------------------------------

fig2, ax2 = plt.subplots(
    figsize=(12, 7),
    facecolor="white"
)

dibujar_barras(ax2)

fig2.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.015),
    ncol=3,
    frameon=False,
    fontsize=9
)

fig2.subplots_adjust(
    left=0.35,
    right=0.97,
    top=0.88,
    bottom=0.25
)

ruta_fig2 = (
    OUTPUT_DIR / "fig_clases_solo_local.png"
)

fig2.savefig(
    ruta_fig2,
    dpi=250,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig2)

# ----------------------------------------------------------
# 11. FIGURA C — COMBINADA 1x2
# ----------------------------------------------------------

fig3 = plt.figure(
    figsize=(18, 9),
    facecolor="white"
)

gs = fig3.add_gridspec(
    1, 2,
    width_ratios=[1, 2.2],
    left=0.06,
    right=0.97,
    top=0.84,
    bottom=0.20,
    wspace=0.60
)

ax3a = fig3.add_subplot(gs[0, 0])
ax3b = fig3.add_subplot(gs[0, 1])

dibujar_matriz(ax3a)
dibujar_barras(ax3b)

fig3.suptitle(
    "Comparación de clasificaciones agrícolas\n"
    "CRCLAC 2020 vs. Land Cover Guatemala 2020",
    fontsize=18,
    fontweight="bold",
    y=0.97
)

fig3.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.04),
    ncol=4,
    frameon=False,
    fontsize=10
)

ruta_fig3 = (
    OUTPUT_DIR / "fig_comparacion_agricola_1x2.png"
)

fig3.savefig(
    ruta_fig3,
    dpi=250,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig3)

# ----------------------------------------------------------
# 12. CONFIRMACIÓN
# ----------------------------------------------------------

print("\n=== FIGURAS GENERADAS ===")

for ruta in [ruta_fig1, ruta_fig2, ruta_fig3]:
    print("[OK]", ruta.name)

print("\nClases locales utilizadas:", len(top))
print("Clases CRCLAC representadas:", len(matriz_top.columns))
print("\n[OK] Visualización finalizada correctamente.")


=== FIGURAS GENERADAS ===
[OK] fig_acuerdo_productos.png
[OK] fig_clases_solo_local.png
[OK] fig_comparacion_agricola_1x2.png

Clases locales utilizadas: 10
Clases CRCLAC representadas: 9

[OK] Visualización finalizada correctamente.
